# Horse Racing Results Predictor

The American professional gambler [Bill Benter](https://en.wikipedia.org/wiki/Bill_Benter) is said to have made earned nearly $1 billion through the development of one of the most successful analysis computer software programs in the horse racing market.

Bill published his techniques in the paper [Computer-Based Horse Race Handicapping and Wagering Systems](https://www.gwern.net/docs/statistics/decision/1994-benter.pdf).

The [YouTube Video by Ken Jee](https://www.youtube.com/watch?v=KEeUR8UDy-s) outlines how he did it, how difficult it was, and discusses whether it is likely to be able to replicate this feat today (hint: Ken thinks it highly unlikely for a number of reasons).

Inspired by video, this notebook examines the possibility of replicating Bill's success using data from modern day UK races.

**NOTE: This is a fun examination of the technique the can be used in predicting races. It is not intended to be accurate or valid. The author accepts no responsibility for the correctness, completeness or quality of the information provided. Please do not use this information to place any real-world bets. Gambling odds are always skewed in favour of the bookmaker and you will lose in the long run.**


In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, date

from race_analytics.features.transforms import (
    surface_categories,
    going_categories,
    race_type_categories,
    encode_surfaces,
    encode_going,
    encode_race_type,
    calculate_speed,
    clean_weight,
    calculate_weight_change,
    calculate_distance_change,
)
from race_analytics.features.race_filters import CalculateRacesWithKnownHorsesAndJockeys
from race_analytics.utils.data_analysis import (
    calculateHorsesPerRace,
    CalculateHorsesStats,
    CalculateJockeyStats,
    CalculateTrainerStats,
)

### Step 1: Load in the historic race data and ignore any horse that didn't complete the race


In [2]:
import glob
import gc
import os
from datetime import timedelta

from race_analytics.data_path import get_data_path

_DATA_PATH = get_data_path()

results_files = glob.glob(os.path.join(_DATA_PATH, "Results_*.csv"))
results_files.sort(reverse=True)  # sort by filename (YYYYMM order) not creation time
results_files = results_files[
    :7
]  # limit result files to current month and previous 6 months to work around memory issue in Azure DevOps
print(results_files)
gc.collect()

['C:\\Dev\\Personal\\RacingData\\Data\\Results_202605.csv', 'C:\\Dev\\Personal\\RacingData\\Data\\Results_202604.csv', 'C:\\Dev\\Personal\\RacingData\\Data\\Results_202603.csv', 'C:\\Dev\\Personal\\RacingData\\Data\\Results_202602.csv', 'C:\\Dev\\Personal\\RacingData\\Data\\Results_202601.csv', 'C:\\Dev\\Personal\\RacingData\\Data\\Results_202512.csv', 'C:\\Dev\\Personal\\RacingData\\Data\\Results_202511.csv']


0

In [3]:
history = pd.concat([pd.read_csv(f) for f in results_files])
history = history[history["ResultStatus"] == "CompletedRace"]
history["Off"] = pd.to_datetime(history["Off"], format="%m/%d/%Y %H:%M:%S")
history["Wins"] = history.apply(
    lambda r: 1 if r["FinishingPosition"] == 1 else 0, axis=1
)

### Analyse factors to understand if they have influence on the outcome of races.

Bill Benter suggested the following attributes:

Current condition:

- performance in recent races
- time since last race
- recent workout data
- age of horse

Past performance:

- finishing position in past races
- lengths behind winner in past races
- normalized times of past races

Adjustments to past performance:

- strength of competition in past races
- weight carried in past races
- jockey's contribution to past performances
- compensation for bad luck in past races
- compensation for advantageous or disadvantageous post position in past races

Present race situational factors:

- weight to be carried
- today's jockey's ability
- advantages or disadvantages of the assigned post position

Preferences which could influence the horse's performance in today's race:

- distance preference
- surface preference (turf vs dirt)
- condition of surface preference (wet vs dry)
- specific track preference


In [4]:
history.columns

Index(['RaceId', 'RaceName', 'CourseId', 'CourseName', 'Off', 'RaceType',
       'Class', 'Pattern', 'RatingBand', 'AgeBand', 'SexRestriction',
       'Distance', 'DistanceInFurlongs', 'DistanceInMeters', 'DistanceInYards',
       'Going', 'Surface', 'HorseId', 'HorseName', 'JockeyId', 'JockeyName',
       'TrainerId', 'TrainerName', 'Age', 'HeadGear', 'RaceCardNumber',
       'StallNumber', 'Weight', 'WeightInPounds', 'FractionalOdds',
       'DecimalOdds', 'OfficialRating', 'RacingPostRating', 'TopSpeedRating',
       'ResultStatus', 'FinishingPosition', 'BeatenDistance',
       'OverallBeatenDistance', 'RaceTime', 'RaceTimeInSeconds', 'Wins'],
      dtype='str')

In [5]:
races = history[
    [
        "RaceId",
        "CourseId",
        "RaceType",
        "Off",
        "DecimalOdds",
        "OfficialRating",
        "RacingPostRating",
        "TopSpeedRating",
        "DistanceInMeters",
        "Going",
        "Surface",
        "HorseId",
        "HorseName",
        "JockeyId",
        "JockeyName",
        "TrainerId",
        "TrainerName",
        "Age",
        "HeadGear",
        "RaceCardNumber",
        "StallNumber",
        "WeightInPounds",
        "FinishingPosition",
        "OverallBeatenDistance",
        "RaceTimeInSeconds",
        "Wins",
    ]
].copy()
history = None
gc.collect()

0

### Expand categorical variables


In [6]:
races = encode_surfaces(races)

In [7]:
races = encode_going(races)

In [8]:
races = encode_race_type(races)

In [9]:
# races["DistanceInMeters"].hist(bins=30)

#### Calculate Horse Stats

Calculate the speed of each horse in the race


In [10]:
races = calculate_speed(races)

In [11]:
races = clean_weight(races)

Calculate the number of horse in each race


In [12]:
races = calculateHorsesPerRace(races)
gc.collect()

0

Feature processors are imported from `utils.data_analysis`.


Calculate, for each race, whether the horse and jockey are known (i.e. have previously been involved in a race)


In [13]:
CalculateRacesWithKnownHorsesAndJockeys().process_race_data(races)
gc.collect()

    CalculateRacesWithKnownHorsesAndJockeys: 202/202 days


0

In [14]:
len(races[races["KnownHorseAndJockey"] == True])

16625

Calculate stats for each horse in the given daily slice based on the history up until that point


In [15]:
CalculateHorsesStats().process_race_data(races)
gc.collect()

    CalculateHorsesStats: 202/202 days


0

In [16]:
races = calculate_weight_change(races)
races = calculate_distance_change(races)

### Evaluating previous horse performance

Now that we have calculated stats for each horse based on the previous races, we can try to figure out if any of the attributes we have relating to previous performance correlate to a horses likelihood of beating another horse in the next race.

Possible performance attributes are:

- Racing post rating
- Official rating
- Top speed rating
- Odds
- Average relative position (i.e. position divided by number of runners in race)


In [17]:
def calculate_correlation_of_race_attribute(attribute: str) -> np.float64:
    head = races[["RaceId", "HorseId", "FinishingPosition", attribute]]
    cross = pd.merge(head, head, on="RaceId")
    head2head = cross[cross["HorseId_x"] > cross["HorseId_y"]].copy().dropna()
    head2head["XBeatsY"] = (
        head2head["FinishingPosition_x"] < head2head["FinishingPosition_y"]
    )
    head2head["Relative"] = head2head[f"{attribute}_x"] - head2head[f"{attribute}_y"]
    return head2head[["XBeatsY", "Relative"]].corr(method="spearman")["XBeatsY"][
        "Relative"
    ]

In [18]:
# rp_corr = calculate_correlation_of_race_attribute("LastRaceRacingPostRating")
# gc.collect()

# or_corr = calculate_correlation_of_race_attribute("LastRaceOfficialRating")
# gc.collect()

# ts_corr = calculate_correlation_of_race_attribute("LastRaceTopSpeedRating")
# gc.collect()

# odds_corr = calculate_correlation_of_race_attribute("LastRaceDecimalOdds")
# gc.collect()
# lr_corr = calculate_correlation_of_race_attribute("LastRaceAvgRelFinishingPosition")
# gc.collect()

# print(f"Correlation for Racing Post Rating {rp_corr}, Official Rating {or_corr}, Top Speed Rating {ts_corr}, Odds {odds_corr}, Avg finish {lr_corr}")

#### Conclusions

Racing post rating most strongly correlates with next race performance, other rating attributes not so much.
Official rating correlation is particularly poor. The problem with all these rating is that there are many rows where the values are undefined.

As expected odds and average finishing position are OK negative correlation indicators of prior performance (i.e. lower values correlate to finishing better). The benefits of these indicators is that they are always available for all horses.

However, all these attributes are actually relatively weak on their own (finishing position being the best with a correlation of -0.249). The predictive power is likely to be in combination with other factors.


#### Calculate Jockey Stats


In [19]:
CalculateTrainerStats().process_race_data(races)
gc.collect()

    CalculateTrainerStats: 202/202 days


0

In [20]:
CalculateJockeyStats().process_race_data(races)
gc.collect()

    CalculateJockeyStats: 202/202 days


0

In [21]:
# wp_corr = calculate_correlation_of_race_attribute("JockeyWinPercentage")
# pp_corr = calculate_correlation_of_race_attribute("JockeyTop3Percentage")
# fp_corr = calculate_correlation_of_race_attribute("JockeyAvgRelFinishingPosition")
# print(f"Correlation for Win percentage {wp_corr}, place percentage {pp_corr}, average finishing position {fp_corr}")

#### Conclusions

Win and place percentage are OK positive correlation indicator of prior performance. Average finishing position is an OK negative correlation indicator of prior performance (i.e. lower values correlate to finishing better).

However, all these attributes are actually relatively weak on there own (finishing position being the best with a correlation of -0.136). The predictive power is likely to be in combination with other factors.


### Examples


In [22]:
# oriental_lilly_races = races[races["HorseId"] == 1439510]
# oriental_lilly_races[["Off", "HorseId", "HorseName", "NumberOfPriorRaces", "FinishingPosition", "DaysRested", "LastRaceAvgRelFinishingPosition"] + [f"LastRace{going}" for going in going_categories]]

In [23]:
# 6901 84857
# james_races = races[(races["JockeyId"] == 6901)].dropna()
# james_races[["Off", "JockeyId", "JockeyName", "JockeyNumberOfPriorRaces", "FinishingPosition", "HorseCount", "JockeyAvgRelFinishingPosition", "JockeyWinPercentage", "JockeyTop3Percentage"]]

### Output


In [24]:
races.info()

<class 'pandas.DataFrame'>
RangeIndex: 83482 entries, 0 to 83481
Data columns (total 81 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   RaceId                           83482 non-null  int64         
 1   CourseId                         83482 non-null  int64         
 2   RaceType                         83482 non-null  str           
 3   Off                              83482 non-null  datetime64[us]
 4   DecimalOdds                      82512 non-null  float64       
 5   OfficialRating                   58043 non-null  float64       
 6   RacingPostRating                 40485 non-null  float64       
 7   TopSpeedRating                   39406 non-null  float64       
 8   DistanceInMeters                 83482 non-null  int64         
 9   Going                            83482 non-null  str           
 10  Surface                          83482 non-null  str           
 11  

In [25]:
races.to_csv(os.path.join(_DATA_PATH, "Race_Features.csv"), index=False)